# Airbnb Amsterdam Data Cleaning & Preparation

This notebook documents the data cleaning and preprocessing steps applied to the Airbnb Amsterdam dataset.

The objective is to transform raw, unstructured data into a clean and consistent dataset suitable for:
- SQL-based analysis
- Power BI dashboard development

The process includes handling missing values, correcting data types, and selecting relevant features for analysis.

## Dataset Overview

The dataset consists of three main files:

- **Listings**: Detailed information about each property and host
- **Reviews**: Customer feedback and review history
- **Calendar**: Availability and pricing over time

Each dataset serves a different purpose and can be combined for deeper analysis.

In [ ]:
import pandas as pd
import datetime

## # Load Data
The datasets includes:
- Listings: detailed property and host information
- Reviews: customer feedback
- Calendar: availability and pricing over time

The datasets are loaded into pandas DataFrames for inspection and preprocessing.

In [ ]:
listings = pd.read_csv('/kaggle/input/datasets/josgomespinheironeto/raw-data/listings.csv')
listings.info()
listings.head()

In [ ]:
reviews = pd.read_csv('/kaggle/input/datasets/josgomespinheironeto/raw-data/reviews.csv')
reviews.info()
reviews.head()

In [ ]:
calendar = pd.read_csv('/kaggle/input/datasets/josgomespinheironeto/raw-data/calendar.csv')
calendar.info()
calendar.head()

### Key Issues Identified

From the initial inspection, the following issues were observed:

- Price stored as string with currency symbols
- Percentage values stored as text
- Boolean values represented as 't'/'f'
- Presence of missing values in key columns
- Large number of irrelevant or redundant columns

## Data Cleaning
### Cleaning Listings Dataset

The original dataset contains over 70 columns, many of which are:
- Redundant
- Irrelevant to the analysis goals

To improve clarity and performance, a subset of key features was selected.

The selected features focus on:
- Pricing (price)
- Availability (availability_365, availability_30, instant_bookable)
- Host characteristics (host_is_superhost, host_response_rate)
- Property attributes (property_type, room_type, accommodates, bedrooms, beds, bathrooms)
- Location (neighbourhood_cleansed, latitude, longitude)
- Performance indicators (number_of_reviews, review_scores_rating)

In [ ]:
columns_to_keep = [
    "id", "host_id",
    "neighbourhood_cleansed", "latitude", "longitude",
    "property_type", "room_type", "accommodates",
    "bedrooms", "beds", "bathrooms",
    "price",
    "availability_365", "availability_30",
    "number_of_reviews", "review_scores_rating",
    "host_is_superhost", "host_response_rate",
    "instant_bookable"
]

listings = listings[columns_to_keep]

### Cleaning Calendar Dataset

The calendar dataset provides daily availability and pricing information for each listing.

During inspection, it was observed that the `price` and `adjusted_price` columns contain only missing values.  
As a result, these fields were excluded from the analysis. `minimum_nights` and `maximum_nights` were also excluded due to holding no real value for the analysis.

The dataset was reduced to retain only relevant columns:
- `listing_id`
- `date`
- `available`

In [ ]:
calendar = calendar[[
    "listing_id",
    "date",
    "available"
]]

### Cleaning Reviews Dataset

The reviews dataset contains individual customer feedback records.

For this project, only structured and relevant fields were retained:
- `listing_id`
- `date`
- `reviewer_id`

The `comments`, `reviewer_id`, and `reviewer_name` fields were excluded, as the analysis focuses on quantitative metrics rather than text-based sentiment and overall user analysis.

In [ ]:
reviews = reviews[[
    "listing_id",
    "date"
]]

### Parsing Date Columns

Date columns in the datasets were converted to datetime64 format to ensure consistency and enable time-based operations.

This transformation allows:
- Proper sorting and filtering
- Time-based aggregations
- Compatibility with SQL and Power BI time functions

In [ ]:
reviews["date_parsed"] = pd.to_datetime(reviews["date"], format="ISO8601")
reviews = reviews[[
    "listing_id",
    "date_parsed"
]]

In [ ]:
calendar["date_parsed"] = pd.to_datetime(calendar["date"], format="ISO8601")
calendar = calendar[[
    "listing_id",
    "date_parsed",
    "available"
]]

### Handling Price Column

The `price` column is stored as a string with currency symbols (e.g., "$132.00"), which prevents numerical operations such as aggregation and comparison. To enable analysis, the column is converted into a numeric format.

In [ ]:
listings["price"] = (
    listings["price"]
    .str.replace(r"[$,]", "", regex=True)
)

listings.price = listings.price.astype("float64")

### Converting Boolean Columns

Some columns (e.g., `host_is_superhost`, `instant_bookable`) are stored as 't'/'f'. These are converted into proper boolean values to improve readability and compatibility with analytical tools such as SQL and Power BI.

In [ ]:
listings["host_is_superhost"] = listings["host_is_superhost"].map({
    "t": True,
    "f": False
})
listings.host_is_superhost = listings.host_is_superhost.astype("boolean")

In [ ]:
listings["instant_bookable"] = listings["instant_bookable"].map({
    "t": True,
    "f": False
})
listings.instant_bookable = listings.instant_bookable.astype("boolean")

In [ ]:
calendar["available"] = calendar["available"].map({
    "t": True,
    "f": False
})
calendar["available"] = calendar["available"].astype("boolean")

### Handling Missing Property Attributes

Some property-related columns, such as `beds` and `bathrooms`, contain missing values.

These values were preserved, as:
- They do not prevent overall analysis
- Removing them would reduce dataset size unnecessarily

The focus is on maintaining dataset integrity while avoiding unnecessary data loss.

In [ ]:
listings["bathrooms"] = listings["bathrooms"].fillna(listings["bathrooms"].median())
listings["beds"] = listings["beds"].fillna(listings["beds"].median())
listings["bedrooms"] = listings["bedrooms"].fillna(listings["bedrooms"].median())

### Converting Percentage Columns

Columns such as `host_response_rate` are stored as strings with percentage symbols (e.g., "100%"), which prevents numerical analysis. These values are converted into numeric format for proper aggregation and comparison.

In [ ]:
listings["host_response_rate"] = (
    listings["host_response_rate"]
    .str.replace("%", "", regex=False)
)
listings.host_response_rate = listings.host_response_rate.astype("float64")

### Removing Listings Without Price

Listings with missing price values were removed from the dataset. Since price is a central variable for this analysis, retaining these records would not contribute to meaningful insights and could distort aggregated metrics.

In [ ]:
listings = listings[listings["price"].notna()]

### Handling Price Outliers

The price distribution revealed extreme values (e.g., listings exceeding 80,000), which are not representative of the typical market. Such outliers can significantly distort averages and mislead analysis. To improve data reliability, extreme values were removed based on statistical thresholds.

In [ ]:
listings = listings[listings["price"] < 1000]

### Creating Derived Metric

An additional feature was created to support downstream analysis:

- **Occupancy Rate**: derived from availability data. It helps to calculate the listings availability throughout the year.

In [ ]:
listings["occupancy_rate"] = 1 - (listings["availability_365"] / 365)

## Final Dataset Overview

The cleaned dataset contains:
- Listings (19 columns & 5.816 rows)
- Calendar (3 columns & 3.825.200 rows)
- Reviews (2 columns & 501.084 rows)
It is now ready for SQL modeling and Power BI visualization.

In [ ]:
listings.to_csv("listings.csv", index=False)
calendar.to_csv("calendar.csv", index=False)
reviews.to_csv("reviews.csv", index=False)

## Next Steps

The cleaned dataset is exported and used for further analysis:

- SQL queries are applied to explore relationships and aggregate metrics
- Power BI is used to build interactive dashboards and extract business insights

This notebook serves as the foundation for the full data analysis workflow.